In [7]:
import requests
import pandas as pd
from time import sleep
from sklearn.preprocessing import StandardScaler

In [8]:
# Team Info (Yahoo Abbreviations)
team_info = {
    108: "la-angels", 109: "arizona", 144: "atlanta", 110: "baltimore", 111: "boston",
    112: "chi-cubs", 113: "cincinnati", 114: "cleveland", 115: "colorado", 116: "detroit",
    117: "houston", 118: "kansas-city", 119: "la-dodgers", 120: "washington", 121: "miami",
    133: "oakland", 134: "pittsburgh", 135: "san-diego", 136: "seattle", 137: "san-francisco",
    138: "st-louis", 139: "tampa-bay", 140: "texas", 141: "toronto", 142: "ny-mets",
    143: "philadelphia", 145: "minnesota", 146: "milwaukee", 147: "ny-yankees"
}
team_ids = list(team_info.keys())

# Headers
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

# Master DataFrames
all_statcast = []
final_merged_df = pd.DataFrame()

# Statcast Features and Betas
hit_features = ['Whiff %','Batted Balls','Hard Hit %','Exit Velocity','XBA','Zone Contact %','Chase %', 'Zone Swing %', 'Straight %']
walk_features = ['Chase %', 'Batted Balls', 'Pitches', 'Whiff %', 'Zone Swing %', 'Zone Contact %']
bats_features = ['Pitches', 'Batted Balls']
bases_features = ['XSLG','Whiff %','Chase %','Barrel %', 'Hard Hit %', 'Exit Velocity', 'Launch Angle', 'Straight %', 'Batted Balls']

hit_betas = {'Whiff %': .6,'Batted Balls': .8,'Hard Hit %': 1,'Exit Velocity': .7,'XBA': .9,'Zone Contact %': .3,'Chase %': .5, 'Zone Swing %': .2, 'Straight %': .4}
walk_betas = {'Chase %': .5, 'Batted Balls': 1, 'Pitches': .8, 'Whiff %': .5, 'Zone Swing %': .1, 'Zone Contact %': .2}
bats_betas = {'Pitches': .3, 'Batted Balls': .7}
bases_betas = {'XSLG': .9,'Whiff %': .5,'Chase %': .5,'Barrel %': 1, 'Hard Hit %': .8, 'Exit Velocity': .7, 'Launch Angle': .3, 'Straight %': .4, 'Batted Balls': .3}

unwanted = ['Unnamed: 1_level_0', 'Unnamed: 0_level_0', 'Statcast', 'Standard Stats','Unnamed: 26_level_1']
team_nicknames = ['D-Backs', 'Braves', 'Orioles', 'Red Sox', 'Cubs', 'White Sox', 'Reds', 'Guardians', 'Rockies', 'Tigers', 'Astros',
                  'Royals', 'Angels', 'Dodgers', 'Marlins', 'Brewers', 'Twins', 'Yankees', 'Athletics', 'Phillies', 'Pirates', 'Padres',
                  'Giants', 'Mariners', 'Cardinals', 'Rays', 'Rangers', 'Blue Jays', 'Nationals', 'Mets']

In [9]:
# Cleaning Data
def clean_column(col):
    for u in unwanted:
        col = col.replace(u, '').strip()
    col = ' '.join(col.split())
    return col

def compute_statcast_score(df, features, betas):
    return sum(df[feat] * betas[feat] for feat in features)

def flip_name(name):
    parts = name.split(',')
    if len(parts) == 2:
        return parts[1].strip() + ' ' + parts[0].strip()
    return name.strip()


In [10]:
# MAIN LOOP
for season in range(2021, 2025):
    print(f"\n=== SEASON: {season} ===")

    for team_id, yah_abbr in team_info.items():
        print(f"Processing: {yah_abbr.upper()} ({season})")

        try:
            url_sav = f"https://baseballsavant.mlb.com/team/{team_id}?view=statcast&nav=hitting&season={season}"
            response = requests.get(url_sav, headers=headers)
            tables = pd.read_html(response.text)
            if len(tables) < 3:
                print(f"{yah_abbr} {season}: Not enough Statcast tables.")
                continue

            standard, statcast1, statcast2 = tables[:3]

            for df in [standard, statcast1, statcast2]:
                df.columns = [clean_column(' '.join(col).strip() if isinstance(col, tuple) else str(col)) for col in df.columns]

            statcast1.drop(columns=[c for c in ['Season','Pitches'] if c in statcast1.columns], inplace=True, errors='ignore')
            statcast2.drop(columns=[c for c in ['Season','Barrel %'] if c in statcast2.columns], inplace=True, errors='ignore')
            statcast3 = standard.drop(columns=[c for c in ['Season','AB','H','2B','3B','HR','BB','PA','SO','BA','OBP','SLG','WOBA','WOBACON'] if c in standard.columns], errors='ignore')
            standard.drop(columns=[c for c in ['Season','PA','SO','BA','OBP','SLG','WOBA','WOBACON','Pitches','Batted Balls','Barrels','Barrel %','Hard Hit %','Exit Velocity','Launch Angle','XBA','XSLG','XWOBA','XWOBACON'] if c in standard.columns], inplace=True, errors='ignore')

            player_col = [col for col in statcast3.columns if 'Player' in col][0]
            statcast3.rename(columns={player_col: 'Player'}, inplace=True)

            statcast = pd.merge(statcast1, statcast2, on='Player')
            statcast = pd.merge(statcast, statcast3, on='Player')

            for col in ['2B', '3B', 'HR', 'H', 'AB', 'BB']:
                if col not in standard.columns:
                    standard[col] = 0

            standard['TB'] = (standard['2B']*2 + standard['3B']*3 + standard['HR']*4 + (standard['H'] - standard['2B'] - standard['3B'] - standard['HR']))
            standard['Runs'] = ((standard['H'] + standard['BB']) * standard['TB']) / (standard['AB'] + standard['BB'])
            standard['Runs'] = standard['Runs'].round()
            standard['TeamID'] = team_id
            standard['Season'] = season
            statcast['TeamID'] = team_id
            statcast['Season'] = season

            all_features = list(set(hit_features + walk_features + bats_features + bases_features))
            missing = [f for f in all_features if f not in statcast.columns]
            if missing:
                print(f"{yah_abbr} {season}: Missing columns: {missing}")
                continue

            scaler = StandardScaler()
            statcast_scaled = statcast.copy()
            statcast_scaled[all_features] = scaler.fit_transform(statcast[all_features])

            statcast['Hit_Score']   = compute_statcast_score(statcast_scaled, hit_features, hit_betas).round(3)
            statcast['Walk_Score']  = compute_statcast_score(statcast_scaled, walk_features, walk_betas).round(3)
            statcast['Bats_Score']  = compute_statcast_score(statcast_scaled, bats_features, bats_betas).round(3)
            statcast['Bases_Score'] = compute_statcast_score(statcast_scaled, bases_features, bases_betas).round(3)

            combined = pd.merge(statcast, standard, on=['Player', 'TeamID', 'Season'], how='inner')
            all_statcast.append(combined)

        except Exception as e:
            print(f"{yah_abbr} {season}: Statcast error - {e}")
            continue

        # Sprinting / Yahoo section (optional per season)
        try:
            tables_sav = pd.read_html(response.text)
            if len(tables_sav) < 7:
                print(f"{yah_abbr} {season}: Sprinting table missing.")
                continue

            sprinting = tables_sav[6]
            tables_yah = pd.read_html(f"https://sports.yahoo.com/mlb/teams/{yah_abbr}/stats/")
            if not tables_yah:
                print(f"{yah_abbr} {season}: Yahoo stats missing.")
                continue

            SB = tables_yah[0]
            merged_rows = []

            for _, sprint_row in sprinting.iterrows():
                matching = SB[SB['Player'] == sprint_row['Player']]
                if not matching.empty:
                    combined = pd.concat([sprint_row, matching.iloc[0].drop('Player')])
                    merged_rows.append(combined)

            if merged_rows:
                merged_df = pd.DataFrame(merged_rows)
                merged_df["Team"] = yah_abbr.upper()
                merged_df["Season"] = season
                to_drop = ['Age', 'Pos Rank', 'Age Rank', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'BB', 'K', 'AVG', 'OBP', 'SLG', 'OPS']
                merged_df.drop(columns=[c for c in to_drop if c in merged_df.columns], inplace=True, errors='ignore')
                final_merged_df = pd.concat([final_merged_df, merged_df], ignore_index=True)

        except Exception as e:
            print(f"{yah_abbr} {season}: Sprint/Yahoo error - {e}")

        sleep(1)


=== SEASON: 2021 ===
Processing: LA-ANGELS (2021)


C:\Users\nncg7\AppData\Local\Temp\ipykernel_16272\3843943379.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_16272\3843943379.py:68: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_sav = pd.read_html(response.text)


Processing: ARIZONA (2021)


C:\Users\nncg7\AppData\Local\Temp\ipykernel_16272\3843943379.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_16272\3843943379.py:68: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_sav = pd.read_html(response.text)


Processing: ATLANTA (2021)


...

C:\Users\nncg7\AppData\Local\Temp\ipykernel_16272\3843943379.py:68: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_sav = pd.read_html(response.text)


Processing: MILWAUKEE (2024)


C:\Users\nncg7\AppData\Local\Temp\ipykernel_16272\3843943379.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
C:\Users\nncg7\AppData\Local\Temp\ipykernel_16272\3843943379.py:68: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_sav = pd.read_html(response.text)


Processing: NY-YANKEES (2024)
ny-yankees 2024: Statcast error - No tables found


C:\Users\nncg7\AppData\Local\Temp\ipykernel_16272\3843943379.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [11]:
# Final processing
df_statcast_all = pd.concat(all_statcast, ignore_index=True)
pattern = r'MLB|' + '|'.join(team_nicknames)
df_statcast_all = df_statcast_all[~df_statcast_all['Player'].str.contains(pattern, case=False, na=False)]
df_statcast_all['Player'] = df_statcast_all['Player'].apply(flip_name)
df_statcast_all['Player'] = df_statcast_all['Player'].str.strip()
final_merged_df['Player'] = final_merged_df['Player'].str.strip()

In [12]:
# Save outputs
df_statcast_all.to_csv('Statcast.csv', index=False)
final_merged_df.to_csv("Sprinting.csv", index=False)

# Join both
merged_df = pd.merge(df_statcast_all, final_merged_df, on='Player', how='inner')
merged_df.to_csv('Statcast Sprinting Merged.csv', index=False)